In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from pyboreas import BoreasDataset
import os

In [2]:
class BoreasRadarDataset(Dataset):
    def __init__(self, data_dir, patches_per_frame=400):
        self.data_dir = Path(data_dir)
        self.patches_per_frame = patches_per_frame
        
        # Grab sorted lists of the batched arrays
        self.patch_files = sorted(list((self.data_dir / "input/patch_arrays").glob("*.npy")))
        self.label_files = sorted(list((self.data_dir / "shifted_labels").glob("*.npy")))
        
        self.num_frames = len(self.patch_files)
        print(f"Found {self.num_frames} frames. Total samples: {self.__len__()}")
        
    def __len__(self):
        return self.num_frames * self.patches_per_frame
        
    def __getitem__(self, idx):
        # Locate the correct file and sub-index
        frame_idx = idx // self.patches_per_frame
        patch_idx = idx % self.patches_per_frame
        
        # Memory map the specific arrays
        batched_patches = np.load(self.patch_files[frame_idx], mmap_mode='r')
        batched_labels = np.load(self.label_files[frame_idx], mmap_mode='r')
        
        # Extract to RAM and convert to 32-bit floats
        # Adding a channel dimension [1, 50, 50] for the CNN
        patch = np.expand_dims(batched_patches[patch_idx].copy(), axis=0).astype(np.float32)
        label = batched_labels[patch_idx].copy().astype(np.float32)
        
        filename = self.patch_files[frame_idx].stem
        return torch.from_numpy(patch), torch.from_numpy(label), filename

In [3]:
class RadarTranslatorCNN(nn.Module):
    def __init__(self, input_size=50, output_bins=6848):
        super().__init__()
        
        # --- ENCODER: Extract 2D Spatial Features ---
        self.encoder = nn.Sequential(
            # Layer 1: [Batch, 1, 50, 50] -> [Batch, 16, 25, 25]
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layer 2: [Batch, 16, 25, 25] -> [Batch, 32, 12, 12]
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layer 3: [Batch, 32, 12, 12] -> [Batch, 64, 6, 6]
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layer 4: [Batch, 64, 6, 6] -> [Batch, 128, 3, 3]
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layer 5: [Batch, 128, 3, 3] -> [Batch, 256, 1, 1]
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Calculate flattened size: 256 channels * 1 height * 1 width = 256
        # The pooling layers have perfectly reduced the spatial dimensions!
        flattened_size = 256
        
        # --- DECODER: Translate to 1D Radar Waveform ---
        self.decoder = nn.Sequential(
            nn.Linear(flattened_size, 1024),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            
            nn.Linear(1024, output_bins),
            # Note: I kept your original ReLU here, but you may want to swap 
            # this for Sigmoid if you are still normalizing your data to 0-1!
            nn.ReLU() 
        )

    def forward(self, x):
        # 1. Extract geometric features
        features = self.encoder(x)
        
        # 2. Flatten spatial dimensions
        features_flat = torch.flatten(features, start_dim=1)
        
        # 3. Predict 1D waveform
        waveform = self.decoder(features_flat)
        
        return waveform

In [ ]:
dataset = BoreasRadarDataset(data_dir="/media/asrl/Extreme SSD/ASRL/boreas/data/boreas-2024-12-03-12-54")
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)
patches, targets, filenames = next(iter(dataloader))
print("Batch shape:", patches.shape)
print("Label shape:", targets.shape)
print(filenames)

In [5]:
sample_patch, sample_label, sample_filename = dataset[0]
num_radar_bins = sample_label.shape[0]

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RadarTranslatorCNN(input_size=50, output_bins=num_radar_bins).to(device)

save_path = "../model_weights/baseline.pth"
state_dict = torch.load(save_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print("Model successfully loaded and ready for inference!")

Model successfully loaded and ready for inference!


In [7]:
azimuths_per_frame = 400
total_frames = len(dataset) // azimuths_per_frame

train_frames = int(0.70 * total_frames) 
valid_frames = int(0.15 * total_frames) 
test_frames = total_frames - train_frames - valid_frames 

# Calculate exact index cutoffs
train_end_idx = train_frames * azimuths_per_frame
valid_end_idx = train_end_idx + (valid_frames * azimuths_per_frame)

# Create chronological subsets that slice cleanly between frames
train_dataset = Subset(dataset, range(0, train_end_idx))
valid_dataset = Subset(dataset, range(train_end_idx, valid_end_idx))

# The test set takes the rest (should naturally be a multiple of azimuths_per_frame)
test_dataset = Subset(dataset, range(valid_end_idx, len(dataset)))

print(f"Total Frames: {total_frames}")
print(f"Frame Split -> Train: {train_frames} | Valid: {valid_frames} | Test: {test_frames}")
print(f"Patch Split -> Train: {len(train_dataset)} | Valid: {len(valid_dataset)} | Test: {len(test_dataset)}")

# Create DataLoaders (Remember to use your worker_init_fn if using multiple workers!)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Total Frames: 141
Frame Split -> Train: 98 | Valid: 21 | Test: 22
Patch Split -> Train: 39200 | Valid: 8400 | Test: 8800


In [ ]:
# ==========================================
# 2. RUN INFERENCE ON THE ENTIRE TEST SET
# ==========================================
print("Running inference on the test set...")
model.eval()
criterion = nn.MSELoss() 

all_test_preds = []
all_test_labels = []
test_loss = 0.0

with torch.no_grad():
    for patches, labels in test_loader:
        patches = patches.to(device)
        
        # --- PREDICTION ---
        # If using the OLD single-head model:
        preds = model(patches)
        
        # If using the NEW Two-Headed model, uncomment this:
        # prob_preds, int_preds = model(patches)
        # preds = prob_preds * int_preds 
        
        # Calculate test loss (using whatever criterion you defined)
        batch_loss = criterion(preds, labels.to(device))
        test_loss += batch_loss.item()
        
        # Move back to CPU and store
        all_test_preds.append(preds.cpu().numpy())
        all_test_labels.append(labels.numpy())

avg_test_loss = test_loss / len(test_loader)
print(f"Average Test Loss: {avg_test_loss:.6f}")

# Concatenate into massive flat arrays of shape (Total_Samples, 6848)
all_test_preds_np = np.concatenate(all_test_preds, axis=0)
all_test_labels_np = np.concatenate(all_test_labels, axis=0)

# ==========================================
# 3. RECONSTRUCT THE FULL POLAR FRAMES
# ==========================================
# Assuming exactly 400 azimuths per frame based on the Boreas dataset
azimuths_per_frame = 400

# Calculate how many full frames we have (drop any weird partial remainders)
num_full_frames = all_test_preds_np.shape[0] // azimuths_per_frame

# Reshape from (N, 6848) to (Frames, 400, 6848)
polar_preds = all_test_preds_np[:num_full_frames * azimuths_per_frame].reshape(num_full_frames, azimuths_per_frame, -1)
polar_labels = all_test_labels_np[:num_full_frames * azimuths_per_frame].reshape(num_full_frames, azimuths_per_frame, -1)

print(f"Successfully reconstructed {num_full_frames} full Polar Radar frames!")
print(f"Polar array shape: {polar_preds.shape}")

# ==========================================
# 4. PLOT 1D WAVEFORM EXAMPLES
# ==========================================
samples_to_plot = 3
fig, axes = plt.subplots(samples_to_plot, 1, figsize=(16, 30), sharex=True)

for i in range(samples_to_plot):
    ax = axes[i]
    ax.plot(all_test_labels_np[i], label="Ground Truth", color='blue', linewidth=1.5)
    ax.plot(all_test_preds_np[i], label="Model Prediction", color='red', linestyle='--', linewidth=1.5)
    
    ax.set_title(f"Test Set - 1D Waveform Sample {i+1}")
    ax.set_ylabel("Reflectivity")
    ax.grid(True, linestyle=':', alpha=0.7)
    if i == 0: ax.legend(loc="upper right")

axes[-1].set_xlabel("Radar Range Bin")
plt.tight_layout()
plt.show()

In [8]:
# Grab the user's home directory
boreas_data = os.getenv("SSD")
boreas_data = os.path.join(boreas_data, "ASRL/boreas/data")
bd = BoreasDataset(boreas_data)

radar_start_frame = 60
radar_end_frame = 200
radar_start_ts = None
radar_end_ts = None

In [9]:
# Run inference on every frame
all_preds = []
all_labels = []
all_filenames = []

with torch.no_grad():   
    for patches, labels, filenames in dataloader:
        patches = patches.to(device)
        preds = model(patches)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
        all_filenames.append(filenames)

all_preds_np = np.concatenate(all_preds, axis=0)
all_labels_np = np.concatenate(all_labels, axis=0)
all_filenames_np = np.concatenate(all_filenames, axis=0)

azimuths_per_frame = 400
num_full_frames = all_preds_np.shape[0] // azimuths_per_frame

# Reshape from (N, 6848) to (Frames, 400, 6848)
polar_preds = all_preds_np[:num_full_frames * azimuths_per_frame].reshape(num_full_frames, azimuths_per_frame, -1)
polar_labels = all_labels_np[:num_full_frames * azimuths_per_frame].reshape(num_full_frames, azimuths_per_frame, -1)
polar_filenames = all_filenames_np[:num_full_frames * azimuths_per_frame].reshape(num_full_frames, azimuths_per_frame)
frame_filenames = polar_filenames[:, 0]

print(f"Successfully reconstructed {num_full_frames} full Polar Radar frames!")
print(f"Polar array shape: {polar_preds.shape}")

Successfully reconstructed 141 full Polar Radar frames!
Polar array shape: (141, 400, 6848)


In [ ]:
for seq in bd.sequences:
    print(f"SequenceID: {seq.ID}")
    radar_start_ts = seq.radar_frames[radar_start_frame].frame
    radar_end_ts = seq.radar_frames[radar_end_frame].frame

    # Reconstruct Cartesian
    for idx in range(radar_start_frame, radar_end_frame + 1):
        i = idx - radar_start_frame
        # if i != 50:
        #     continue

        radar_frame = seq.get_radar(idx)
        assert(frame_filenames[i] == radar_frame.frame)
        radar_frame.polar = polar_labels[i]
        radar_frame.visualize()
        radar_frame.polar = polar_preds[i]
        radar_frame.visualize()

        radar_frame.unload_data()
        # break
    break

In [14]:
print(frame_filenames.shape)

(141,)
